# 05 LLM 训练出现 Loss Spike，如何诊断与恢复？

## 面试回答主线

Loss spike 不是单一根因；先区分数据异常、数值溢出、梯度/参数范数突然增大、学习率切换、通信错误和 checkpoint 损坏。诊断必须同时看 loss、梯度范数、参数范数、token/数据来源、AMP overflow 和各 rank 差异。恢复策略通常是保存最近健康 checkpoint、隔离异常 batch、降低或回退学习率、启用梯度裁剪，并在相同数据切片上复跑确认。实验人为注入一个异常学习率，比较裸更新与“梯度裁剪 + health checkpoint 回滚”。它演示控制面机制，不声称覆盖真实分布式故障。

**核心公式：** 常用门限是 $\lVert g_t\rVert_2 > r_t+\delta$，其中 $r_t$ 是滑动均值；若候选更新后的 $L_{t+1}>\tau L_t$，可拒绝该更新并回退到健康状态。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
class TicketNet(nn.Module):  # 定义一个显式 forward 的极小工单分类网络。
    def __init__(self):  # 初始化可训练矩阵。
        super().__init__()  # 初始化模块基类。
        self.weight = nn.Parameter(torch.randn(3, 2) * 0.15)  # 创建分类权重。
    def forward(self, batch):  # 写出从特征到 logits 的直接映射。
        return batch @ self.weight  # 返回分类 logits。
def raw_train(schedule):  # 在给定学习率表下训练且记录诊断量。
    model = TicketNet()  # 创建独立模型避免污染其他对照。
    losses = []  # 保存每步 loss。
    grad_norms = []  # 保存每步梯度范数。
    for learning_rate in schedule:  # 逐步读取学习率事件。
        loss = torch.nn.functional.cross_entropy(model(features), labels)  # 计算全量教学批损失。
        loss.backward()  # 计算梯度。
        grad_norm = float(model.weight.grad.norm())  # 记录更新前梯度范数。
        with torch.no_grad():  # 手写 SGD 更新。
            if learning_rate > 1.0:  # 用异常学习率事件模拟错误的更新符号。
                model.weight += learning_rate * model.weight.grad  # 模拟溢出或通信损坏导致的反向更新。
            else:  # 正常学习率事件使用下降方向。
                model.weight -= learning_rate * model.weight.grad  # 按当前学习率更新参数。
            model.weight.grad.zero_()  # 清除梯度以免累积。
        losses.append(float(loss))  # 保存损失。
        grad_norms.append(grad_norm)  # 保存梯度范数。
    return losses, grad_norms  # 返回完整诊断轨迹。
unsafe_schedule = [0.18, 0.18, 0.18, 18.0, 0.18, 0.18]  # 注入一个异常大的学习率事件。
raw_losses, raw_grad_norms = raw_train(unsafe_schedule)  # 运行没有保护的基线。
baseline_metric = max(raw_losses)  # 保存裸训练的最大 loss。
print(f'裸训练 loss={ [round(value, 3) for value in raw_losses] }，梯度范数={ [round(value, 3) for value in raw_grad_norms] }')  # 输出诊断曲线。


裸训练 loss=[0.632, 0.609, 0.587, 0.566, 4.365, 4.259]，梯度范数=[0.361, 0.35, 0.34, 0.33, 0.767, 0.766]


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
def guarded_train(schedule):  # 实现带裁剪、候选检查和回滚的训练控制面。
    model = TicketNet()  # 创建受保护的独立模型。
    accepted_losses = []  # 保存被接受的健康损失。
    incident_count = 0  # 统计拒绝的异常更新数。
    for learning_rate in schedule:  # 逐步执行可能异常的学习率事件。
        current_loss = torch.nn.functional.cross_entropy(model(features), labels)  # 计算当前健康损失。
        current_loss.backward()  # 获得候选更新的梯度。
        healthy_weight = model.weight.detach().clone()  # 保存可回滚的健康权重。
        gradient = model.weight.grad.detach().clone()  # 复制梯度供裁剪和诊断。
        clipped_gradient = gradient * min(1.0, 1.0 / (float(gradient.norm()) + 1e-6))  # 手写全局范数裁剪。
        with torch.no_grad():  # 在不构图环境中试运行候选更新。
            if learning_rate > 1.0:  # 让健康门限面对同一种异常符号损坏。
                model.weight += learning_rate * clipped_gradient  # 写入被故意损坏方向的候选参数。
            else:  # 正常事件采用下降方向。
                model.weight -= learning_rate * clipped_gradient  # 写入正常候选参数。
        candidate_loss = torch.nn.functional.cross_entropy(model(features), labels)  # 检查候选更新后的损失。
        if float(candidate_loss) > float(current_loss) * 1.45:  # 用相对 loss 门限识别异常候选。
            with torch.no_grad():  # 进入回滚环境。
                model.weight.copy_(healthy_weight)  # 恢复最近健康权重。
            incident_count += 1  # 记录一次被拒绝的更新。
            accepted_losses.append(float(current_loss))  # 保留健康状态的损失。
        else:  # 候选通过健康门限时。
            accepted_losses.append(float(candidate_loss))  # 接受更新后的损失。
        model.weight.grad.zero_()  # 清除当前步梯度。
    return accepted_losses, incident_count  # 返回健康轨迹和事故数。
safe_losses, incidents = guarded_train(unsafe_schedule)  # 在同一异常 schedule 上运行保护策略。
core_metric = max(safe_losses)  # 记录保护后的最大 loss。
print(f'保护训练 loss={ [round(value, 3) for value in safe_losses] }，拒绝更新数={incidents}')  # 展示恢复控制面的真实行为。


保护训练 loss=[0.639, 0.613, 0.588, 0.588, 0.565, 0.544]，拒绝更新数=1


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=4.365134
核心机制     | 指标=0.639386


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **Loss Spike** 的关键状态与更新路径。生产恢复必须保存优化器状态、随机数状态、数据游标与并行拓扑；只恢复模型权重可能马上复发。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
failure_metric = raw_losses[4]  # 选择异常学习率之后的裸训练损失作为失败证据。
fix_metric = safe_losses[4]  # 选择同一步的保护训练损失作为修复证据。
print(f'失败定位：异常步后裸 loss={failure_metric:.3f}；回滚与裁剪后 loss={fix_metric:.3f}')  # 对齐同一时刻比较恢复效果。


失败定位：异常步后裸 loss=4.365；回滚与裁剪后 loss=0.565


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产恢复必须保存优化器状态、随机数状态、数据游标与并行拓扑；只恢复模型权重可能马上复发。

**常见坑：** 看到单步尖峰就盲目回滚，可能把正常的噪声误判为事故；也不能只依赖 clip 掩盖持续性数据或硬件问题。

**延伸追问：** 如何判定是单 rank 溢出还是全局数据问题？健康 checkpoint 的保留频率如何平衡存储成本与 RPO？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert incidents >= 1  # 验证健康门限确实拦截了异常候选更新。
assert core_metric < baseline_metric  # 验证保护策略降低了最大 loss。
assert fix_metric < failure_metric  # 验证异常步后的恢复优于裸训练。
assert len(raw_losses) == len(safe_losses)  # 验证两组比较使用同一训练步数。
